In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
# loading the data set

df = pd.read_csv("../data/hotel_bookings 2.csv")

In [12]:
df.columns

Index(['hotel', 'is_canceled', 'lead_time', 'arrival_date_year',
       'arrival_date_month', 'arrival_date_week_number',
       'arrival_date_day_of_month', 'stays_in_weekend_nights',
       'stays_in_week_nights', 'adults', 'children', 'babies', 'meal',
       'country', 'market_segment', 'distribution_channel',
       'is_repeated_guest', 'previous_cancellations',
       'previous_bookings_not_canceled', 'reserved_room_type',
       'assigned_room_type', 'booking_changes', 'deposit_type', 'agent',
       'company', 'days_in_waiting_list', 'customer_type', 'adr',
       'required_car_parking_spaces', 'total_of_special_requests',
       'total_guests', 'total_stay_nights', 'has_booking_changes'],
      dtype='object')

In [7]:
# removing the data leakage

df = df.drop(
    columns=["reservation_status", "reservation_status_date"]
)

In [8]:
df["total_guests"] = (
    df["adults"] +
    df["children"].fillna(0) +
    df["babies"]
)

In [9]:
df["total_stay_nights"] = (
    df["stays_in_weekend_nights"] +
    df["stays_in_week_nights"]
)

In [10]:
df["has_booking_changes"] = (
    df["booking_changes"] > 0
).astype(int)

In [13]:
df["total_previous_bookings"] = (
    df["previous_cancellations"] +
    df["previous_bookings_not_canceled"]
)

In [14]:
df["previous_cancellation_rate"] = np.where(
    df["total_previous_bookings"] > 0,
    df["previous_cancellations"] / df["total_previous_bookings"],
    0
)

In [18]:
df["room_changed"] = (
    df["reserved_room_type"] !=
    df["assigned_room_type"]
).astype(int)

In [19]:
df["agent_missing"] = df["agent"].isna().astype(int)

df["company_missing"] = df["company"].isna().astype(int)

In [20]:
df = df.drop(columns=["agent", "company"])

In [21]:
df["children"] = df["children"].fillna(0)

In [22]:
df["country"] = df["country"].fillna("Unknown")

In [23]:
months = {
    "January": 1,
    "February": 2,
    "March": 3,
    "April": 4,
    "May": 5,
    "June": 6,
    "July": 7,
    "August": 8,
    "September": 9,
    "October": 10,
    "November": 11,
    "December": 12
}

df["arrival_month_num"] = df["arrival_date_month"].map(months)

In [24]:
df["estimated_revenue"] = (
    df["adr"] *
    df["total_stay_nights"]
)

In [25]:
df[df["adr"] < 0]

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,total_guests,total_stay_nights,has_booking_changes,total_previous_bookings,previous_cancellation_rate,room_changed,agent_missing,company_missing,arrival_month_num,estimated_revenue
14969,Resort Hotel,0,195,2017,March,10,5,4,6,2,...,2.0,10,1,2,0.0,1,0,1,3,-63.8


In [26]:
df[df["total_guests"] == 0]

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,total_guests,total_stay_nights,has_booking_changes,total_previous_bookings,previous_cancellation_rate,room_changed,agent_missing,company_missing,arrival_month_num,estimated_revenue
2224,Resort Hotel,0,1,2015,October,41,6,0,3,0,...,0.0,3,1,0,0.0,1,1,0,10,0.00
2409,Resort Hotel,0,0,2015,October,42,12,0,0,0,...,0.0,0,0,0,0.0,1,1,0,10,0.00
3181,Resort Hotel,0,36,2015,November,47,20,1,2,0,...,0.0,3,0,0,0.0,1,0,1,11,0.00
3684,Resort Hotel,0,165,2015,December,53,30,1,4,0,...,0.0,5,1,0,0.0,0,0,1,12,0.00
3708,Resort Hotel,0,165,2015,December,53,30,2,4,0,...,0.0,6,1,0,0.0,1,0,1,12,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115029,City Hotel,0,107,2017,June,26,27,0,3,0,...,0.0,3,1,0,0.0,0,0,1,6,302.40
115091,City Hotel,0,1,2017,June,26,30,0,1,0,...,0.0,1,0,0,0.0,1,1,1,6,0.00
116251,City Hotel,0,44,2017,July,28,15,1,1,0,...,0.0,2,1,0,0.0,1,0,1,7,147.60
116534,City Hotel,0,2,2017,July,28,15,2,5,0,...,0.0,7,1,0,0.0,1,0,1,7,160.02


In [27]:
from sklearn.model_selection import train_test_split

X = df.drop(columns="is_canceled")
y = df["is_canceled"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [28]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

In [29]:
categorical_features = X_train.select_dtypes(
    include="object"
).columns

numerical_features = X_train.select_dtypes(
    exclude="object"
).columns

In [30]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        ),
        (
            "num",
            "passthrough",
            numerical_features
        )
    ]
)

In [31]:
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000
    ))
])

logistic_model.fit(X_train, y_train)

/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['hotel', 'arrival_date_month', 'meal', 'country', 'market_segment',
       'distribution_channel', 'reserved_room_type', 'assigned_room_type',
       'deposit_type', 'customer_type'],
      dtype='object')),
                                                 ('num', 'passthrough',
                                                  Index(['lead_time', 'arrival_date_ye...
       'days_in_waiting_list', 'adr', 'required_car_parking_spaces',
       'total_of_special_requests', 'total_guests', 'total_stay_nights',
       'has_booking_changes', 'total_previous_bookings',
       'previous_cancellation_rate', 'room_changed', 'agent_missing',
       'company_missing', 'arrival_month_num', 'estimated_revenue'],
      dtype='object'))])),
                ('model', LogisticRegression(max_iter=1000))])

In [32]:
y_pred = logistic_model.predict(X_test)
y_prob = logistic_model.predict_proba(X_test)[:, 1]

In [33]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Accuracy: 0.8112907278666555
Precision: 0.8133757041744908
Recall: 0.6366308648954211
F1: 0.7142313546423136
ROC-AUC: 0.8883914141479662


In [34]:
from sklearn.ensemble import RandomForestClassifier

In [35]:
from xgboost import XGBClassifier

In [36]:
xgb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42
    ))
])

In [37]:
xgb_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['hotel', 'arrival_date_month', 'meal', 'country', 'market_segment',
       'distribution_channel', 'reserved_room_type', 'assigned_room_type',
       'deposit_type', 'customer_type'],
      dtype='object')),
                                                 ('num', 'passthrough',
                                                  Index(['lead_time', 'arrival_date_ye...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.05,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=6, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=400, n_jobs=None,
                               num_parallel_tree=None, ...))])